In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import tt_ind_solve_power
from statsmodels.stats.power import tt_ind_solve_power

Вы аналитик в команде монетизации классифайда. Бы проведен тест с экранами платных услуг в монетизации. Основная гипотеза что мы сможем увеличить ARPU на +3.5 рубля.
У нас есть данные и мы хотим проанализировать этот тест.
Для этого у нас есть данные по продавцам авто участвующим в тесте stratification_homework_2.csv

**Схема файла следующая**
- ARPU: целевая мтерика в тесте
- is_capital: продавец авто из столицы (capital) или из регионов (region)
- is_pro: профессиональный продавец? 1 - профи, 0 - простой
- group: Группа в Аб-тесте. (1 - тестовая, 0 - контрольная)

Задание 1. Нужно провести базовый t-test и сделать вывод по его результатам. Найдите и ввести полученное p-value

Задание 2. Необходимо найти наилучшую для разбиения страту. Для этого нужно рассчитать стратифицированную дисперсию при примении обоих страт и указать ту, при которой дисперсия сокращается максимально.

Задание 3. Применить постстратификацию к данным и пересчитать результаты t-test. Указать p-value при применении наилучшей страты.

Задание 4. Взяв за основу данные по контрольной группе (group == 0), рассчитать нужный размер выборки для детекции наблюдаемой абсолютной разницы средних между тестовой и контрольной группами. Рассчитать размер группы для простого t-test и стратифицированного t-test.

# Загружаем данные

In [ ]:
data = pd.read_csv('stratification_homework_2.csv')

In [ ]:
data.head(10)

,ARPU,is_capital,is_pro,group
0,250.5,region,1,1
1,182.0,capital,0,1
2,75.0,region,0,0
3,532.5,capital,1,0
4,88.0,region,0,1
5,111.0,region,0,1
6,86.0,region,0,1
7,193.0,capital,0,1
8,173.0,region,0,0
9,105.0,region,0,0


# создадим нужные функции для проведения стратифицированного анализа

In [ ]:
def calculate_strat_mean(data, strata_name, gen_pop_weights, target_value):
    ''' Вычисляет стратифицированное среднее. использует веса из генеральной популяции (передаются как словарь)'''
    strats_means = data.groupby(strata_name)[target_value].mean()

    data_means_weights = pd.merge(
        pd.Series(strats_means, name='value_means'),
        pd.Series(gen_pop_weights, name='weight'),
        how='inner',
        left_index=True,
        right_index=True)

    mean_strat = (data_means_weights['weight'] * data_means_weights['value_means']).sum()
    return(mean_strat)

In [ ]:
def calculate_mean(values):
    '''Вычисляет обычное среднее'''
    return(np.mean(values))

In [ ]:
def calculate_strat_var(data, strata_name, gen_pop_weights, target_value):
    '''Вычисляет стратифицированную дисперсию.'''

    strat_vars = data.groupby(strata_name)[target_value].var()

    data_vars_weights = pd.merge(
        pd.Series(strat_vars, name='value_vars'),
        pd.Series(gen_pop_weights, name='weight'),
        how='inner',
        left_index=True,
        right_index=True)

    var_strat = (data_vars_weights['weight'] * data_vars_weights['value_vars']).sum()
    return (var_strat)

In [ ]:
# Перепишем фунции для проведения t-test/ Будем использовать

def get_basic_ttest(group_A, group_B):
    '''Проверяет гипотезу о равенстве средних для обычного среднего.
    return - t_stat, p_value.'''

    t_stat, p_value = stats.ttest_ind(group_A, group_B)
    inference = {'t_stat': t_stat, 'p_value':p_value}
    return(inference)


def get_strat_ttest(df_A, df_B, strata_name, target_value, gen_pop_weights):
    """Проверяет гипотезу о равенстве средних для стратифицированного среднего и стратифицировнной дисперсии.

    return - pvalue.
    """
    mean_strat_A =  calculate_strat_mean(df_A, strata_name, gen_pop_weights, target_value)
    mean_strat_B =  calculate_strat_mean(df_B, strata_name, gen_pop_weights, target_value)

    var_strat_A = calculate_strat_var(df_A, strata_name, gen_pop_weights, target_value)
    var_strat_B = calculate_strat_var(df_B, strata_name, gen_pop_weights, target_value)

    delta_mean_strat = mean_strat_A - mean_strat_B
    std_mean_strat = np.sqrt(var_strat_A / len(df_A) + var_strat_B / len(df_B))
    t_stat_strat = delta_mean_strat / std_mean_strat
    p_value = 2 * (1 - stats.norm.cdf(np.abs(t_stat_strat)))

    inference = {'t_stat': t_stat_strat, 'p_value':p_value}
    return (inference)

## Задание 1. Проведите простой t-test для нестратифицированных данных

In [ ]:
# проведем простой t-test. Найдем p_value
inference = get_basic_ttest(data['ARPU'][data['group']==0], data['ARPU'][data['group']==1])
print('p-value базового t-test = ', round(inference['p_value'], 3))

p-value базового t-test =  0.065


## Задание 2. Найдите оптимальную страту для разбиения
Сравните дисперсию метрики простую и стратифицированную в контрольной группе
Найдите по какой страте мы получаем наибольшую разницу между дисперсиями в двух группах

In [ ]:
strat_list = ['is_capital', 'is_pro']

In [ ]:
# Задание 1. Найдем по какой ковариате у нас сильнее всего сокращается дисперсия если мы считаем стратифицированную дисперсию

start_disp_decr_list = []
gen_var = round(data['ARPU'].var())
gen_pop_weights_list = []
print('Глобальная дисперсия данных', gen_var)
for strat in strat_list:
    strat_name = data[strat].unique()
    gen_pop_weights = {strat_name[1]:data[strat].value_counts(normalize = True)[strat_name[1]], strat_name[0]:data[strat].value_counts(normalize = True)[strat_name[0]]}
    gen_pop_weights_list.append(gen_pop_weights)
    strat_var = round(calculate_strat_var(data, strat, gen_pop_weights, 'ARPU'))
    disp_decr = round(strat_var/gen_var - 1, 3)
    start_disp_decr_list.append(disp_decr)
    print('Стратифицироанная дисперсия для страты {} = {}. Дисперсия сокращается на {}%'.format(strat, strat_var, disp_decr*100))
print('Больше всего дисперсия из за стратификации сокращается по страте {}. Сокращается на {}%.'.format(strat_list[np.argmin(start_disp_decr_list)], np.min(start_disp_decr_list)*100))

Глобальная дисперсия данных 7480
Стратифицироанная дисперсия для страты is_capital = 5790. Дисперсия сокращается на -22.6%
Стратифицироанная дисперсия для страты is_pro = 6194. Дисперсия сокращается на -17.2%
Больше всего дисперсия из за стратификации сокращается по страте is_capital. Сокращается на -22.6%.


# Задание 3. Проведите стратифицированный t-test по наилучшей страте полученной в задании 1

In [ ]:
# проведем стратифицированный t-test
inference = get_strat_ttest(data[data['group']==0], data[data['group']==1], 'is_capital', 'ARPU', gen_pop_weights_list[0])
print('p-value базового t-test = ', round(inference['p_value'], 3))

p-value базового t-test =  0.018


# Задание 4.
### Взяв за основу контрольную группу (group = 0) найдите размер выборки, который нужен был бы, чтобы найти  абсолютный MDE = 3 при обычном и стратифицированном критерии.
- Альфа = 0.05
- beta = 0.2
- критерий двусторонний
- выборка 50/50


In [ ]:
#зададим исходные распределения
abs_MDE = data['ARPU'][data['group']==1].mean() - data['ARPU'][data['group']==0].mean()
print('Абсолютный MDE =', abs_MDE)

# работаем только с группой A
data_a = data[data['group']==0]
group_a_weights = dict(data_a['is_capital'].value_counts(normalize = True))
group_a_weights

basic_mean = data_a['ARPU'].mean()
basic_std = data_a['ARPU'].std()

# рассчитаем стратифицированное стандартное отклонение
strat_std = calculate_strat_var(data_a, 'is_capital', group_a_weights, 'ARPU') ** 0.5

print('Обычная станд. отклонение =', basic_std)
print('Стратифицированное станд. отклонение =', strat_std)

basic_cohen_D_effect_size = abs_MDE / basic_std
strat_cohen_D_effect_size = abs_MDE / strat_std

print('basic Cohen D effect size = ', basic_cohen_D_effect_size)
print('strat Cohen D effect size = ', strat_cohen_D_effect_size)

strat_sample_size = int(tt_ind_solve_power(effect_size = strat_cohen_D_effect_size, alpha=0.05, power=0.8, nobs1=None, ratio=1))
basic_sample_size = int(tt_ind_solve_power(effect_size = basic_cohen_D_effect_size, alpha=0.05, power=0.8, nobs1=None, ratio=1))
print('Ответ--------------------------------')
print('Basic sample size = {}'.format(basic_sample_size))
print('Stratified sample size = {}'.format(strat_sample_size))

Абсолютный MDE = 3.2421437067065995
Обычная станд. отклонение = 87.24597419606758
Стратифицированное станд. отклонение = 76.1470117161366
basic Cohen D effect size =  0.0371609548358133
strat Cohen D effect size =  0.042577425346549014
Ответ--------------------------------
Basic sample size = 11368
Stratified sample size = 8660
